In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pickle
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from statsmodels.stats.outliers_influence import variance_inflation_factor

from Modules.panel_utils import (
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
)
from Modules.test_fun import *
from Modules.model_results_export import (
    ModelResultsAggregator, 
    ensure_results_dir, add_model_set, 
    build_and_export
)
import warnings
warnings.filterwarnings('ignore')


#New_Loans_Progr_DOMRF

In [2]:
###############################
    #ЗАГРУЗКА + ПРЕДОБРАБОТКА ДАННЫХ
###############################
df_exog = pd.read_excel('Operations/exog_reg_analys.xlsx') 
df_exog = df_exog.drop(['Unnamed: 0'], axis=1)

In [3]:
# ============ ЛОГАРИФМИРОВАНИЕ ============ #

loan_vars = ['New_Loans_Progr_DOMRF'] 

for var in loan_vars:
    if var in df_exog.columns:
        df_exog[f'ln_{var}'] = np.log(df_exog[var])
        
ln_loan_vars = [f'ln_{var}' for var in loan_vars if f'ln_{var}' in df_exog.columns]


# ============ СОЗДАНИЕ ПЕРВЫХ РАЗНОСТЕЙ  ============ #

# Разности для логарифмов 
for var in ln_loan_vars:
    d_var = f'd_{var}'
    df_exog[d_var] = df_exog.groupby('Region')[var].diff()

if 'd_ln_New_Loans_Progr_DOMRF' in df_exog.columns:
    df_exog['d_ln_New_Loans_Progr_DOMRF_lag1'] = df_exog['d_ln_New_Loans_Progr_DOMRF'].shift(1)
df_exog['ln_Fin_Dostup_lag1'] = df_exog['ln_Fin_Dostup'].shift(1)


KeyError: 'd_ln_New_Loans_Progr_DOMRF'

In [4]:
lag_vars = [
            'MaP_Fact',
            'Mortgage_MaP_Fact',
            'Conscred_MaP_Fact',
            'MaP_Tight_Fact',
            'MaP_Ease_Fact',
            'Mortgage_MaP_Tight_Fact',
            'Mortgage_MaP_Ease_Fact',
            'Conscred_MaP_Tight_Fact',
            'Conscred_MaP_Ease_Fact'
           ]

for var in lag_vars:
    for i in range(12):
        lag_vars = f'{var}_lag{i+1}'
        df_exog[lag_vars] = df_exog[var].shift(i+1) 

lag_vars_mon = [
            'd_ROISFIX_pos',
            'd_ROISFIX_neg',
            'd_MIACR_pos',
            'd_MIACR_neg',
            'd_Mon_Shock_neg',
            'd_Mon_Shock_pos',
           ]

for var in lag_vars_mon:
    for i in range(12):
        lag_var_mon = f'{var}_lag{i+1}'
        df_exog[lag_var_mon] = df_exog[var].shift(i+1) 

lag_vars_new_loans = [
            'd_ln_New_Loans_Fl',
            'd_ln_New_Loans_Mort',
            'd_ln_New_Loans_ConsCred',
           ]

for var in lag_vars_new_loans:
    for i in range(12):
        lag_vars_new_loans = f'{var}_lag{i+1}'
        df_exog[lag_vars_new_loans] = df_exog[var].shift(i+1)    

 
df_exog['ln_Fin_Dostup_lag1'] = df_exog['ln_Fin_Dostup'].shift(1)  
#  Создаем взаимодействия для переменных макропру
map_var_makro = [
    #'MaP_Fact',
    #'MaP_Tight_Fact',
    #'MaP_Ease_Fact',
    # 'Conscred_MaP_Fact',
    #'Conscred_MaP_Tight_Fact',
    #'Conscred_MaP_Ease_Fact',
    'Mortgage_MaP_Fact',
    #'Mortgage_MaP_Tight_Fact',
    #'Mortgage_MaP_Ease_Fact',
    #'MaP_Announcement',
    # 'Conscred_MaP_Announcement',
    'Mortgage_MaP_Announcement',
    #'MaP_Tight_Announcement',
    #'MaP_Ease_Announcement',
    #'Conscred_MaP_Tight_Announcement', 
    #'Conscred_MaP_Ease_Announcement',
    #'Mortgage_MaP_Tight_Announcement', 
    #'Mortgage_MaP_Ease_Announcement'
]

z_vars_makro = [
    'ln_Fin_Dostup_lag1',
    'Cap_to_assets_lag1',
    'Cluster_new_cd_1',
    'Cluster_new_cd_3',
    'Cluster_new_cd_4',
    'd_Mon_Shock_neg',
    'd_Mon_Shock_pos'
]

for m in map_var_makro:
    for z in z_vars_makro:
        col_name = f"{m}_{z}"
        df_exog[col_name] = df_exog[m] * df_exog[z]
        
lag_interaction = [
    # 'MaP_Fact_ln_Fin_Dostup_lag1',
    # 'MaP_Fact_Cap_to_assets_lag1',
    # 'MaP_Fact_Cluster_new_cd_1',
    # 'MaP_Fact_Cluster_new_cd_3',
    # 'MaP_Fact_Cluster_new_cd_4',    
    # 'MaP_Fact_d_Mon_Shock_neg',
    # 'MaP_Fact_d_Mon_Shock_pos',
    
    # 'MaP_Tight_Fact_ln_Fin_Dostup_lag1',
    # 'MaP_Tight_Fact_Cap_to_assets_lag1',
    # 'MaP_Tight_Fact_Cluster_new_cd_1',
    # 'MaP_Tight_Fact_Cluster_new_cd_3',
    # 'MaP_Tight_Fact_Cluster_new_cd_4',
    # 'MaP_Tight_Fact_d_Mon_Shock_neg',
    # 'MaP_Tight_Fact_d_Mon_Shock_pos',
    
    # 'MaP_Ease_Fact_ln_Fin_Dostup_lag1',
    # 'MaP_Ease_Fact_Cap_to_assets_lag1',
    # 'MaP_Ease_Fact_Cluster_new_cd_1',
    # 'MaP_Ease_Fact_Cluster_new_cd_3',
    # 'MaP_Ease_Fact_Cluster_new_cd_4',
    # 'MaP_Ease_Fact_d_Mon_Shock_neg',
    # 'MaP_Ease_Fact_d_Mon_Shock_pos',
    
    # 'Conscred_MaP_Fact_ln_Fin_Dostup_lag1',
    # 'Conscred_MaP_Fact_Cap_to_assets_lag1',
    # 'Conscred_MaP_Fact_Cluster_new_cd_1',
    # 'Conscred_MaP_Fact_Cluster_new_cd_3',
    # 'Conscred_MaP_Fact_Cluster_new_cd_4',
    # 'Conscred_MaP_Fact_d_Mon_Shock_neg',
    # 'Conscred_MaP_Fact_d_Mon_Shock_pos',
    
    # 'Conscred_MaP_Tight_Fact_ln_Fin_Dostup_lag1',
    # 'Conscred_MaP_Tight_Fact_Cap_to_assets_lag1',
    # 'Conscred_MaP_Tight_Fact_Cluster_new_cd_1',
    # 'Conscred_MaP_Tight_Fact_Cluster_new_cd_3',
    # 'Conscred_MaP_Tight_Fact_Cluster_new_cd_4',
    # 'Conscred_MaP_Tight_Fact_d_Mon_Shock_neg',
    # 'Conscred_MaP_Tight_Fact_d_Mon_Shock_pos',
    
    # 'Conscred_MaP_Ease_Fact_ln_Fin_Dostup_lag1',
    # 'Conscred_MaP_Ease_Fact_Cap_to_assets_lag1',
    # 'Conscred_MaP_Ease_Fact_Cluster_new_cd_1',
    # 'Conscred_MaP_Ease_Fact_Cluster_new_cd_3',
    # 'Conscred_MaP_Ease_Fact_Cluster_new_cd_4',
    # 'Conscred_MaP_Ease_Fact_d_Mon_Shock_neg',
    # 'Conscred_MaP_Ease_Fact_d_Mon_Shock_pos',
    
    'Mortgage_MaP_Fact_ln_Fin_Dostup_lag1',
    'Mortgage_MaP_Fact_Cap_to_assets_lag1',
    'Mortgage_MaP_Fact_Cluster_new_cd_1',
    'Mortgage_MaP_Fact_Cluster_new_cd_3',
    'Mortgage_MaP_Fact_Cluster_new_cd_4',
    'Mortgage_MaP_Fact_d_Mon_Shock_neg',
    'Mortgage_MaP_Fact_d_Mon_Shock_pos',
    
    # 'Mortgage_MaP_Tight_Fact_ln_Fin_Dostup_lag1',
    # 'Mortgage_MaP_Tight_Fact_Cap_to_assets_lag1',
    # 'Mortgage_MaP_Tight_Fact_Cluster_new_cd_1',
    # 'Mortgage_MaP_Tight_Fact_Cluster_new_cd_3',
    # 'Mortgage_MaP_Tight_Fact_Cluster_new_cd_4',
    # 'Mortgage_MaP_Tight_Fact_d_Mon_Shock_neg',
    # 'Mortgage_MaP_Tight_Fact_d_Mon_Shock_pos',
    
    # 'Mortgage_MaP_Ease_Fact_ln_Fin_Dostup_lag1',
    # 'Mortgage_MaP_Ease_Fact_Cap_to_assets_lag1',
    # 'Mortgage_MaP_Ease_Fact_Cluster_new_cd_1',
    # 'Mortgage_MaP_Ease_Fact_Cluster_new_cd_3',
    # 'Mortgage_MaP_Ease_Fact_Cluster_new_cd_4',
    # 'Mortgage_MaP_Ease_Fact_d_Mon_Shock_neg',
    # 'Mortgage_MaP_Ease_Fact_d_Mon_Shock_pos',
    
    # 'MaP_Announcement_d_Mon_Shock_neg',
    # 'MaP_Announcement_d_Mon_Shock_pos',
    
     # 'Conscred_MaP_Announcement_d_Mon_Shock_neg',
     # 'Conscred_MaP_Announcement_d_Mon_Shock_pos',
    
    'Mortgage_MaP_Announcement_d_Mon_Shock_neg',
    'Mortgage_MaP_Announcement_d_Mon_Shock_pos',

    # 'MaP_Tight_Announcement_d_Mon_Shock_neg',
    # 'MaP_Tight_Announcementt_d_Mon_Shock_pos',
    # 'MaP_Ease_Announcement_d_Mon_Shock_neg',
    # 'MaP_Ease_Announcementt_d_Mon_Shock_pos',

    
    # 'Conscred_MaP_Tight_Announcement_d_Mon_Shock_neg',
    # 'Conscred_MaP_Tight_Announcement_d_Mon_Shock_pos',
    # 'Conscred_MaP_Ease_Announcement_d_Mon_Shock_neg',
    # 'Conscred_MaP_Ease_Announcement_d_Mon_Shock_pos',
    
    # 'Mortgage_MaP_Tight_Announcement_d_Mon_Shock_neg',
    # 'Mortgage_MaP_Tight_Announcement_d_Mon_Shock_pos',
    # 'Mortgage_MaP_Ease_Announcement_d_Mon_Shock_neg',
    # 'Mortgage_MaP_Ease_Announcement_d_Mon_Shock_pos',
           ]

for var in lag_interaction:
    for i in range(12):
        lag_interaction = f'{var}_lag{i+1}'
        df_exog[lag_interaction] = df_exog[var].shift(i+1)    

# Лаги переменной анонса МакПру (ипотека)
for i in range(12):
    df_exog[f'Mortgage_MaP_Announcement_lag{i+1}'] = df_exog['Mortgage_MaP_Announcement'].shift(i+1)

df_exog = df_exog.dropna()

df_exog = df_exog.query(
    "Date < '2026-01-01'"
).reset_index(drop=True)

df_exog_pre_sank = df_exog[df_exog['Sank_dum']==0]
df_exog_post_sank = df_exog[df_exog['Sank_dum']==1]

df_exog_post_sank_new = df_exog.query(
    "Date >= '2023-01-01'"
).reset_index(drop=True)

### Создание модели

In [5]:
df_test = df_exog.copy()

In [ ]:
###############################
# Зависимая переменная, базовые регрессоры, доп. импорты
###############################

from Modules.panel_utils import collect_all_test_pvalues, run_spec_tests_robust
from Modules.model_results_export import build_and_export_with_tests

dependent_var = 'd_ln_New_Loans_Mort'

exog_vars_base = [
    'd_ln_New_Loans_Mort_lag1',
    'ln_Fin_Dostup_lag1',
    'Def_Zadolg_Fl_lag1',
    'Cap_to_assets_lag1',
    'CPI_reg_lag1',
    'Oil_p',
    'REER',
]

cluster_dummies = ['Cluster_new_cd_1', 'Cluster_new_cd_3', 'Cluster_new_cd_4']

## Блок 1 — Гетерогенность МакПру × Кластеры

Спецификации лаг 0–12. Каждая: `Mortgage_MaP_Fact_lagN` + кластерные дамми + взаимодействия `Mortgage_MaP_Fact_lagN × Cluster_lagN`.

In [ ]:
###############################
# Блок 1 — построение спецификаций (лаг 0–12)
###############################

shock_vars_b1 = []
for n in range(13):
    map_var = 'Mortgage_MaP_Fact' if n == 0 else f'Mortgage_MaP_Fact_lag{n}'
    int_cd1 = 'Mortgage_MaP_Fact_Cluster_new_cd_1' if n == 0 else f'Mortgage_MaP_Fact_Cluster_new_cd_1_lag{n}'
    int_cd3 = 'Mortgage_MaP_Fact_Cluster_new_cd_3' if n == 0 else f'Mortgage_MaP_Fact_Cluster_new_cd_3_lag{n}'
    int_cd4 = 'Mortgage_MaP_Fact_Cluster_new_cd_4' if n == 0 else f'Mortgage_MaP_Fact_Cluster_new_cd_4_lag{n}'
    shock_vars_b1.append([map_var] + cluster_dummies + [int_cd1, int_cd3, int_cd4])

model_spec_b1 = build_shock_variants(dependent_var, exog_vars_base, shock_vars_b1)
exog_variants_b1 = model_spec_b1['exog_variants']
print(f'Блок 1: {len(exog_variants_b1)} спецификаций')

In [ ]:
###############################
# Блок 1 — оценка моделей и сбор тестов
###############################

model_specs_b1 = []
tests_by_column_b1 = {}

for idx, exog_vars in enumerate(exog_variants_b1):
    spec_name = f'Лаг {idx}'

    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_test, df_test, dependent_var, exog_vars,
        cov_type='dk', cluster_entity=True, df_name='df_test'
    )

    model_specs_b1.append({
        'spec_name': spec_name,
        'dependent_var': dependent_var,
        'subsample': 'Общая выборка',
        're': re_res if re_success else None,
    })

    tests = collect_all_test_pvalues(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success,
        shock_vars=shock_vars_b1[idx]
    )
    _, hausman_pval_r, _, bp_lm_pval_r, _, f_pval_r = run_spec_tests_robust(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success
    )
    spec_tests = {**tests.get('spec_tests', {}), **{
        'Hausman (FE vs RE) p-value (robust)': hausman_pval_r,
        'Breusch-Pagan LM (RE vs Pooled) p-value (robust)': bp_lm_pval_r,
        'F-test (FE vs Pooled) p-value (robust)': f_pval_r,
    }}
    diag_re = tests.get('diagnostics', {}).get('RE', {})
    tests_by_column_b1[f'{spec_name} (RE)'] = {**spec_tests, **diag_re}

    print(f'  {spec_name} — RE: {"OK" if re_success else "FAIL"}')

print('Блок 1 оценён.')

## Блок 2 — Гетерогенность МакПру × Монетарный шок

Спецификации лаг 0–12. Каждая: `Mortgage_MaP_Fact_lagN` + `d_Mon_Shock_pos/neg_lagN` + кластерные дамми + взаимодействия `Mortgage_MaP_Fact_lagN × d_Mon_Shock_pos/neg_lagN`.

In [ ]:
###############################
# Блок 2 — построение спецификаций (лаг 0–12)
###############################

shock_vars_b2 = []
for n in range(13):
    map_var = 'Mortgage_MaP_Fact' if n == 0 else f'Mortgage_MaP_Fact_lag{n}'
    ms_pos  = 'd_Mon_Shock_pos' if n == 0 else f'd_Mon_Shock_pos_lag{n}'
    ms_neg  = 'd_Mon_Shock_neg' if n == 0 else f'd_Mon_Shock_neg_lag{n}'
    int_pos = 'Mortgage_MaP_Fact_d_Mon_Shock_pos' if n == 0 else f'Mortgage_MaP_Fact_d_Mon_Shock_pos_lag{n}'
    int_neg = 'Mortgage_MaP_Fact_d_Mon_Shock_neg' if n == 0 else f'Mortgage_MaP_Fact_d_Mon_Shock_neg_lag{n}'
    shock_vars_b2.append([map_var, ms_pos, ms_neg] + cluster_dummies + [int_pos, int_neg])

model_spec_b2 = build_shock_variants(dependent_var, exog_vars_base, shock_vars_b2)
exog_variants_b2 = model_spec_b2['exog_variants']
print(f'Блок 2: {len(exog_variants_b2)} спецификаций')

In [ ]:
###############################
# Блок 2 — оценка моделей и сбор тестов
###############################

model_specs_b2 = []
tests_by_column_b2 = {}

for idx, exog_vars in enumerate(exog_variants_b2):
    spec_name = f'Лаг {idx}'

    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_test, df_test, dependent_var, exog_vars,
        cov_type='dk', cluster_entity=True, df_name='df_test'
    )

    model_specs_b2.append({
        'spec_name': spec_name,
        'dependent_var': dependent_var,
        'subsample': 'Общая выборка',
        're': re_res if re_success else None,
    })

    tests = collect_all_test_pvalues(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success,
        shock_vars=shock_vars_b2[idx]
    )
    _, hausman_pval_r, _, bp_lm_pval_r, _, f_pval_r = run_spec_tests_robust(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success
    )
    spec_tests = {**tests.get('spec_tests', {}), **{
        'Hausman (FE vs RE) p-value (robust)': hausman_pval_r,
        'Breusch-Pagan LM (RE vs Pooled) p-value (robust)': bp_lm_pval_r,
        'F-test (FE vs Pooled) p-value (robust)': f_pval_r,
    }}
    diag_re = tests.get('diagnostics', {}).get('RE', {})
    tests_by_column_b2[f'{spec_name} (RE)'] = {**spec_tests, **diag_re}

    print(f'  {spec_name} — RE: {"OK" if re_success else "FAIL"}')

print('Блок 2 оценён.')

## Блок 3 — МакПру Анонс (лаг 0–12)

Спецификации лаг 0–12. Каждая: `Mortgage_MaP_Announcement_lagN` + кластерные дамми.

In [ ]:
###############################
# Блок 3 — построение спецификаций (лаг 0–12)
###############################

shock_vars_b3 = []
for n in range(13):
    ann_var = 'Mortgage_MaP_Announcement' if n == 0 else f'Mortgage_MaP_Announcement_lag{n}'
    shock_vars_b3.append([ann_var] + cluster_dummies)

model_spec_b3 = build_shock_variants(dependent_var, exog_vars_base, shock_vars_b3)
exog_variants_b3 = model_spec_b3['exog_variants']
print(f'Блок 3: {len(exog_variants_b3)} спецификаций')

In [ ]:
###############################
# Блок 3 — оценка моделей и сбор тестов
###############################

model_specs_b3 = []
tests_by_column_b3 = {}

for idx, exog_vars in enumerate(exog_variants_b3):
    spec_name = f'Лаг {idx}'

    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
        df_test, df_test, dependent_var, exog_vars,
        cov_type='dk', cluster_entity=True, df_name='df_test'
    )

    model_specs_b3.append({
        'spec_name': spec_name,
        'dependent_var': dependent_var,
        'subsample': 'Общая выборка',
        're': re_res if re_success else None,
    })

    tests = collect_all_test_pvalues(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success,
        shock_vars=shock_vars_b3[idx]
    )
    _, hausman_pval_r, _, bp_lm_pval_r, _, f_pval_r = run_spec_tests_robust(
        y, X, pooled_res, fe_res, re_res,
        pooled_success, fe_success, re_success
    )
    spec_tests = {**tests.get('spec_tests', {}), **{
        'Hausman (FE vs RE) p-value (robust)': hausman_pval_r,
        'Breusch-Pagan LM (RE vs Pooled) p-value (robust)': bp_lm_pval_r,
        'F-test (FE vs Pooled) p-value (robust)': f_pval_r,
    }}
    diag_re = tests.get('diagnostics', {}).get('RE', {})
    tests_by_column_b3[f'{spec_name} (RE)'] = {**spec_tests, **diag_re}

    print(f'  {spec_name} — RE: {"OK" if re_success else "FAIL"}')

print('Блок 3 оценён.')

In [ ]:
import re
###############################
# Экспорт RE — оба блока в один файл, два листа
###############################

results_dir = ensure_results_dir('Results')
date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
out_path = os.path.join(results_dir, f'Mort_heterogen_{date_tag}.xlsx')


def _build_export_df(aggregator, tests_by_col, decimals=3, test_decimals=6):
    tbl = aggregator.build_table(include_pvalues=True, decimals=decimals,
                                   var_rename_fn=lambda v: re.sub(r'_lag\d+$', '', v))
    col_names = list(tbl.columns)
    test_names = list(dict.fromkeys(n for col in col_names for n in tests_by_col.get(col, {})))
    test_rows = {}
    for tn in test_names:
        row = {}
        for col in col_names:
            v = tests_by_col.get(col, {}).get(tn)
            row[col] = '' if v is None or (isinstance(v, float) and pd.isna(v)) else f'{float(v):.{test_decimals}f}'
        test_rows[tn] = row
    tests_df = pd.DataFrame.from_dict(test_rows, orient='index').reindex(columns=col_names).fillna('')
    blank = pd.DataFrame([[''] * len(col_names)], index=[''], columns=col_names)
    header = pd.DataFrame([[''] * len(col_names)], index=['ТЕСТЫ'], columns=col_names)
    return pd.concat([tbl, blank, header, tests_df])


aggregator_b1 = ModelResultsAggregator()
for spec in model_specs_b1:
    if spec['re'] is not None:
        aggregator_b1.add_model_results(
            spec['re'], spec['dependent_var'], spec['subsample'],
            'RE', spec['spec_name'], se_type='dk'
        )

aggregator_b2 = ModelResultsAggregator()
for spec in model_specs_b2:
    if spec['re'] is not None:
        aggregator_b2.add_model_results(
            spec['re'], spec['dependent_var'], spec['subsample'],
            'RE', spec['spec_name'], se_type='dk'
        )

aggregator_b3 = ModelResultsAggregator()
for spec in model_specs_b3:
    if spec['re'] is not None:
        aggregator_b3.add_model_results(
            spec['re'], spec['dependent_var'], spec['subsample'],
            'RE', spec['spec_name'], se_type='dk'
        )

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    _build_export_df(aggregator_b1, tests_by_column_b1).to_excel(writer, sheet_name='МакПру_Кластеры')
    _build_export_df(aggregator_b2, tests_by_column_b2).to_excel(writer, sheet_name='МакПру_МонШок')
    _build_export_df(aggregator_b3, tests_by_column_b3).to_excel(writer, sheet_name='МакПру_Анонс')

print(f'Экспортировано → {out_path}')